# 03 — Evaluating the enterprise search engine

> **Applied Labs** · **Domain**: RAG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/12_enterprise_search/03_evaluate.ipynb)

**Prerequisites**:
- [02_build.ipynb](./02_build.ipynb) — Working enterprise search engine with hybrid retrieval and RAG

**What you will learn**:
- Define and compute retrieval metrics: MRR, NDCG, Recall@k
- Compare BM25 vs dense vs hybrid retrieval quantitatively
- Use LLM-as-judge to evaluate answer faithfulness and relevance
- Measure citation accuracy: precision and recall of source attribution
- Profile latency and cost per query at scale
- Categorize failure modes and build an improvement roadmap

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0"

import os, re, json, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set
from collections import Counter, defaultdict

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY", "sk-REPLACE-ME"))
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — Evaluation framework

We evaluate the enterprise search engine across **three dimensions**:

| Dimension | Metrics | What it measures |
|-----------|---------|-----------------|
| **Retrieval** | MRR@k, NDCG@k, Recall@k | Does the retriever surface the right chunks? |
| **Generation** | Faithfulness, Relevance | Is the answer correct and grounded? |
| **Citation** | Citation precision, Citation recall | Are sources correctly attributed? |

Let's define the test dataset and metric functions.

In [ ]:
@dataclass
class EvalQuery:
    """A test query with ground-truth annotations."""
    query_id: str
    query: str
    relevant_doc_ids: List[str]      # gold-standard relevant documents
    expected_answer_keywords: List[str]  # key facts the answer should contain
    category: str                     # factual, multi-source, comparison, multi-hop, abstention


# 10-query evaluation set with ground truth
EVAL_SET: List[EvalQuery] = [
    EvalQuery("q01", "What is the PTO policy for full-time employees?",
              ["W02"], ["15 days", "roll over", "5 days", "2 weeks", "HR portal"], "factual"),
    EvalQuery("q02", "How do I process a customer refund?",
              ["D05", "W01"], ["eligibility", "CRM", "Stripe", "3-5 business days"], "factual"),
    EvalQuery("q03", "What happened with the billing error for customer #4521?",
              ["E02", "C01"], ["double charge", "$340", "idempotency", "BIL-2847"], "multi-source"),
    EvalQuery("q04", "What are the Q4 product priorities?",
              ["D02", "C03"], ["SSO", "Widget Pro", "Q4", "SAML"], "multi-source"),
    EvalQuery("q05", "How do I deploy a service to production?",
              ["D01"], ["Helm", "canary", "5%", "rollback", "namespace"], "factual"),
    EvalQuery("q06", "What is the data retention policy for GDPR compliance?",
              ["W06"], ["3 years", "anonymized", "30 days", "7 years"], "factual"),
    EvalQuery("q07", "Compare the expense and PTO approval processes.",
              ["D08", "W02"], ["manager approval", "$200", "2 weeks", "Expensify"], "comparison"),
    EvalQuery("q08", "What SSO features are planned and when?",
              ["C03", "D02"], ["SAML 2.0", "OIDC", "Q4", "v2.1"], "multi-hop"),
    EvalQuery("q09", "What is the company's cryptocurrency policy?",
              [], [], "abstention"),
    EvalQuery("q10", "What are the code review requirements before merging?",
              ["W07"], ["one approval", "tests", "security", "N+1"], "factual"),
]

print(f"Evaluation set: {len(EVAL_SET)} queries\n")
cat_counts = Counter(q.category for q in EVAL_SET)
for cat, count in cat_counts.most_common():
    print(f"  {cat:<15} {count} queries")

## Section 2 — Retrieval quality

We compute three standard retrieval metrics:
- **MRR@k** (Mean Reciprocal Rank) — How high does the first relevant doc rank?
- **NDCG@k** (Normalized Discounted Cumulative Gain) — Rank-weighted relevance
- **Recall@k** — What fraction of relevant docs appear in top-k?

We compare **BM25**, **dense**, and **hybrid** retrieval side by side.

In [ ]:
def mrr_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int = 10) -> float:
    """Mean Reciprocal Rank at k."""
    for rank, doc_id in enumerate(retrieved_ids[:k], 1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def recall_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int = 5) -> float:
    """Recall at k — fraction of relevant docs in top-k results."""
    if not relevant_ids:
        return 1.0  # abstention queries
    retrieved_set = set(retrieved_ids[:k])
    return len(retrieved_set & relevant_ids) / len(relevant_ids)


def ndcg_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int = 10) -> float:
    """Normalized Discounted Cumulative Gain at k."""
    dcg = 0.0
    for i, doc_id in enumerate(retrieved_ids[:k]):
        rel = 1.0 if doc_id in relevant_ids else 0.0
        dcg += rel / math.log2(i + 2)  # i+2 because log2(1) = 0
    # Ideal DCG
    ideal_rels = sorted([1.0] * min(len(relevant_ids), k) +
                        [0.0] * max(0, k - len(relevant_ids)), reverse=True)
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal_rels))
    return dcg / idcg if idcg > 0 else 0.0


# Simulate retrieval results for each method
# (In a real eval, we'd run actual retrieval — here we simulate plausible rankings)

def simulate_bm25_ranking(query: EvalQuery) -> List[str]:
    """Simulate BM25 results — good at exact keyword match, bad at synonyms."""
    np.random.seed(hash(query.query_id) % 2**31)
    all_doc_ids = [f"W0{i}" for i in range(1, 9)] + [f"D0{i}" for i in range(1, 9)] + \
                  [f"C0{i}" for i in range(1, 5)] + [f"E0{i}" for i in range(1, 5)]
    relevant = set(query.relevant_doc_ids)
    # BM25: relevant docs appear but often not at top
    others = [d for d in all_doc_ids if d not in relevant]
    np.random.shuffle(others)
    result = list(relevant) + others
    # Insert 1-3 irrelevant docs before relevant ones (simulating BM25 weakness)
    if relevant:
        insert_count = min(2, len(others))
        for i in range(insert_count):
            result.insert(i, others[i])
    return result[:10]


def simulate_dense_ranking(query: EvalQuery) -> List[str]:
    """Simulate dense retrieval — better at semantics, may miss exact matches."""
    np.random.seed(hash(query.query_id) % 2**31 + 1)
    all_doc_ids = [f"W0{i}" for i in range(1, 9)] + [f"D0{i}" for i in range(1, 9)] + \
                  [f"C0{i}" for i in range(1, 5)] + [f"E0{i}" for i in range(1, 5)]
    relevant = set(query.relevant_doc_ids)
    others = [d for d in all_doc_ids if d not in relevant]
    np.random.shuffle(others)
    # Dense: relevant docs usually near top
    result = list(relevant) + others
    if relevant and np.random.random() < 0.3:
        result.insert(0, others[0])
    return result[:10]


def simulate_hybrid_ranking(query: EvalQuery) -> List[str]:
    """Simulate hybrid retrieval — best of both worlds."""
    relevant = list(query.relevant_doc_ids)
    np.random.seed(hash(query.query_id) % 2**31 + 2)
    all_doc_ids = [f"W0{i}" for i in range(1, 9)] + [f"D0{i}" for i in range(1, 9)] + \
                  [f"C0{i}" for i in range(1, 5)] + [f"E0{i}" for i in range(1, 5)]
    others = [d for d in all_doc_ids if d not in set(relevant)]
    np.random.shuffle(others)
    # Hybrid: relevant docs at top with high probability
    return (relevant + others)[:10]


# Compute metrics
methods = {
    "BM25":   simulate_bm25_ranking,
    "Dense":  simulate_dense_ranking,
    "Hybrid": simulate_hybrid_ranking,
}

results_table: List[Dict] = []
for method_name, ranker_fn in methods.items():
    mrr_scores, recall_scores, ndcg_scores = [], [], []
    for eq in EVAL_SET:
        if not eq.relevant_doc_ids:
            continue  # skip abstention queries for retrieval eval
        ranking = ranker_fn(eq)
        relevant = set(eq.relevant_doc_ids)
        mrr_scores.append(mrr_at_k(ranking, relevant, k=10))
        recall_scores.append(recall_at_k(ranking, relevant, k=5))
        ndcg_scores.append(ndcg_at_k(ranking, relevant, k=10))

    results_table.append({
        "Method": method_name,
        "MRR@10": f"{np.mean(mrr_scores):.3f}",
        "Recall@5": f"{np.mean(recall_scores):.3f}",
        "NDCG@10": f"{np.mean(ndcg_scores):.3f}",
    })

df_metrics = pd.DataFrame(results_table)
print("Retrieval Quality Comparison")
print("=" * 50)
print(df_metrics.to_string(index=False))
print("\n→ Hybrid retrieval consistently outperforms both BM25 and dense alone.")

## Section 3 — Answer quality (LLM-as-judge)

We use an **LLM-as-judge** approach: a separate LLM call evaluates each answer
on two dimensions:
- **Faithfulness** (1-5): Is every claim supported by the provided context?
- **Relevance** (1-5): Does the answer actually address the question?

This avoids expensive human evaluation while providing calibrated scores.

In [ ]:
JUDGE_PROMPT = """You are an impartial evaluator of search engine answers.
Rate the following answer on two dimensions. Return JSON only.

Dimensions:
- faithfulness (1-5): Is every claim in the answer supported by the context? 5 = fully grounded, 1 = fabricated.
- relevance (1-5): Does the answer address the user's question? 5 = perfectly relevant, 1 = off-topic.

Context chunks:
{context}

Question: {question}

Answer: {answer}

Return ONLY valid JSON: {{"faithfulness": <int>, "relevance": <int>, "reasoning": "<brief explanation>"}}"""


def judge_answer(
    question: str,
    answer: str,
    context: str,
    model: str = CHAT_MODEL,
) -> Dict:
    """Use LLM-as-judge to evaluate answer quality.

    Args:
        question: The original query.
        answer: The generated answer.
        context: The context chunks used for generation.
        model: Judge model.

    Returns:
        Dict with faithfulness, relevance scores, and reasoning.
    """
    prompt = JUDGE_PROMPT.format(context=context, question=question, answer=answer)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=200,
    )
    text = response.choices[0].message.content.strip()
    try:
        # Extract JSON from response
        json_match = re.search(r'\{[^}]+\}', text)
        if json_match:
            return json.loads(json_match.group())
        return json.loads(text)
    except json.JSONDecodeError:
        return {"faithfulness": 0, "relevance": 0, "reasoning": f"Parse error: {text[:100]}"}


# Simulated answers for evaluation (in production, use actual RAG output)
SIMULATED_ANSWERS = [
    {"query": EVAL_SET[0].query,
     "answer": "Full-time employees earn 15 days of PTO per year [1]. Unused PTO rolls over up to 5 days [1]. Requests must be submitted 2 weeks in advance via the HR portal [1].",
     "context": "Full-time employees earn 15 days paid time off per year. PTO accrues at 1.25 days per month. Unused PTO rolls over up to 5 days. Requests must be submitted at least 2 weeks in advance via the HR portal."},
    {"query": EVAL_SET[1].query,
     "answer": "To process a refund: verify eligibility per Return & Exchange Guidelines [1], log the request in CRM [1], process via Stripe payment gateway [1], and send a confirmation email [1]. The SLA is 3-5 business days [1].",
     "context": "Step 1: Verify refund eligibility per Return & Exchange Guidelines. Step 2: Log refund request in CRM with reason code. Step 3: Process refund via Stripe payment gateway. Step 4: Send confirmation email. Processing SLA: 3-5 business days."},
    {"query": EVAL_SET[4].query,
     "answer": "Deploy to the production k8s cluster using Helm charts [1]. Ensure namespace isolation [1]. Run a canary deploy to 5% traffic first [1], then monitor error rates for 15 minutes [1]. Rollback with helm rollback if needed [1].",
     "context": "Deploy services to the production k8s cluster using Helm charts. Ensure namespace isolation between staging and production. Run canary deploy to 5% traffic. Monitor error rates for 15 minutes. Rollback: helm rollback <release>."},
    {"query": EVAL_SET[8].query,
     "answer": "I don't have enough information to answer this question. The provided context does not contain any policy regarding cryptocurrency investments for the corporate treasury.",
     "context": "No relevant context found."},
]

print("Answer Quality Evaluation (LLM-as-Judge)")
print("=" * 70)
judge_results: List[Dict] = []
for sa in SIMULATED_ANSWERS:
    scores = judge_answer(sa["query"], sa["answer"], sa["context"])
    judge_results.append({
        "Query": sa["query"][:45] + "...",
        "Faithfulness": scores.get("faithfulness", "?"),
        "Relevance": scores.get("relevance", "?"),
        "Reasoning": scores.get("reasoning", "")[:60],
    })
    print(f"Q: {sa['query'][:60]}...")
    print(f"   Faithfulness: {scores.get('faithfulness', '?')}/5  "
          f"Relevance: {scores.get('relevance', '?')}/5")
    print(f"   {scores.get('reasoning', '')[:80]}")
    print()

df_judge = pd.DataFrame(judge_results)
print(df_judge[["Query", "Faithfulness", "Relevance"]].to_string(index=False))

## Section 4 — Citation accuracy

Citations are the trust foundation of enterprise search. We measure:
- **Citation precision**: Of all citations in the answer, how many actually support the claim?
- **Citation recall**: Of all facts in the answer, how many have a citation?

A system with high faithfulness but low citation recall gives correct answers
without proof — users can't verify claims.

In [ ]:
def extract_citations(answer: str) -> List[int]:
    """Extract citation numbers [n] from an answer string."""
    return [int(m) for m in re.findall(r'\[(\d+)\]', answer)]


def citation_precision_recall(
    answer: str,
    n_context_chunks: int,
    n_claims: int,
) -> Dict[str, float]:
    """Compute citation precision and recall.

    Args:
        answer: The generated answer with [n] citations.
        n_context_chunks: Number of context chunks provided.
        n_claims: Number of factual claims in the answer (human-annotated).

    Returns:
        Dict with precision and recall.
    """
    citations = extract_citations(answer)
    unique_citations = set(citations)

    # Precision: are all cited chunks actually in the provided context?
    valid_citations = {c for c in unique_citations if 1 <= c <= n_context_chunks}
    precision = len(valid_citations) / len(unique_citations) if unique_citations else 1.0

    # Recall: do all claims have a citation?
    recall = len(citations) / n_claims if n_claims > 0 else 1.0

    return {
        "n_citations": len(citations),
        "unique_refs": len(unique_citations),
        "valid_refs": len(valid_citations),
        "precision": precision,
        "recall": min(recall, 1.0),
    }


# Evaluate citations on our simulated answers
CITATION_ANNOTATIONS = [
    {"answer": SIMULATED_ANSWERS[0]["answer"], "n_chunks": 3, "n_claims": 3},
    {"answer": SIMULATED_ANSWERS[1]["answer"], "n_chunks": 3, "n_claims": 5},
    {"answer": SIMULATED_ANSWERS[2]["answer"], "n_chunks": 3, "n_claims": 5},
    {"answer": "The PTO policy gives 15 days. Rollover is 5 days.",
     "n_chunks": 3, "n_claims": 2},  # Missing citations!
]

print("Citation Accuracy Analysis")
print("=" * 70)
rows: List[Dict] = []
for i, ann in enumerate(CITATION_ANNOTATIONS, 1):
    metrics = citation_precision_recall(ann["answer"], ann["n_chunks"], ann["n_claims"])
    label = "well-cited" if metrics["precision"] >= 0.8 and metrics["recall"] >= 0.8 else "needs improvement"
    rows.append({
        "Answer": f"A{i}",
        "Citations": metrics["n_citations"],
        "Valid refs": metrics["valid_refs"],
        "Precision": f"{metrics['precision']:.2f}",
        "Recall": f"{metrics['recall']:.2f}",
        "Verdict": label,
    })

df_cite = pd.DataFrame(rows)
print(df_cite.to_string(index=False))
print("\n→ A4 demonstrates the 'uncited claims' failure — correct answer but no citations.")
print("  Users cannot verify the claims, reducing trust in the system.")

## Section 5 — Latency and cost

Enterprise search must be **fast** (< 3s end-to-end) and **cost-efficient**
at scale. We profile each pipeline stage and project costs.

In [ ]:
# Latency breakdown (simulated realistic numbers)
LATENCY_PROFILE = {
    "Query embedding": 80,     # ms
    "Dense retrieval": 25,     # ms (vector similarity)
    "BM25 retrieval": 15,      # ms (inverted index)
    "RRF fusion": 2,           # ms
    "LLM generation": 1200,    # ms (gpt-4o-mini, ~300 tokens)
    "Total": 1322,
}

print("Per-Query Latency Breakdown")
print("=" * 50)
for stage, ms in LATENCY_PROFILE.items():
    bar = "█" * (ms // 20)
    print(f"  {stage:<20} {ms:>6} ms  {bar}")

# Cost projection
print("\n\nCost Projection at Scale")
print("=" * 50)
QUERIES_PER_DAY = 10_000
EMBEDDING_COST_PER_1K = 0.00002   # text-embedding-3-small
CHAT_COST_INPUT_1K = 0.00015      # gpt-4o-mini input
CHAT_COST_OUTPUT_1K = 0.0006      # gpt-4o-mini output
AVG_INPUT_TOKENS = 800
AVG_OUTPUT_TOKENS = 200

daily_embed_cost = QUERIES_PER_DAY * (AVG_INPUT_TOKENS / 1000) * EMBEDDING_COST_PER_1K
daily_chat_cost = QUERIES_PER_DAY * (
    (AVG_INPUT_TOKENS / 1000) * CHAT_COST_INPUT_1K +
    (AVG_OUTPUT_TOKENS / 1000) * CHAT_COST_OUTPUT_1K
)
daily_total = daily_embed_cost + daily_chat_cost
monthly_total = daily_total * 30

costs = {
    "Embeddings (daily)": daily_embed_cost,
    "Chat/gen (daily)": daily_chat_cost,
    "Total daily": daily_total,
    "Total monthly": monthly_total,
    "Cost per query": daily_total / QUERIES_PER_DAY,
}

for label, cost in costs.items():
    print(f"  {label:<22} ${cost:.4f}" if cost < 1 else f"  {label:<22} ${cost:.2f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Latency pie
stages = list(LATENCY_PROFILE.keys())[:-1]
times = [LATENCY_PROFILE[s] for s in stages]
colors_lat = ["#3498db", "#2ecc71", "#27ae60", "#95a5a6", "#e74c3c"]
axes[0].pie(times, labels=stages, autopct="%1.0f%%", colors=colors_lat, startangle=90)
axes[0].set_title("Latency Breakdown (per query)")

# Cost bar
cost_labels = ["Embedding", "Generation"]
cost_values = [daily_embed_cost * 30, daily_chat_cost * 30]
axes[1].bar(cost_labels, cost_values, color=["#3498db", "#e74c3c"], edgecolor="white")
axes[1].set_ylabel("$ / month")
axes[1].set_title(f"Monthly Cost at {QUERIES_PER_DAY:,} queries/day")
for i, v in enumerate(cost_values):
    axes[1].text(i, v + 0.5, f"${v:.2f}", ha="center", fontweight="bold")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## Section 6 — Failure analysis

Every search system has failure modes. Categorizing them drives the improvement
roadmap. We analyze failures across retrieval, generation, and citation.

In [ ]:
# Failure taxonomy
@dataclass
class FailureCase:
    """A categorized search failure."""
    category: str
    subcategory: str
    description: str
    example_query: str
    impact: str       # high, medium, low
    fix_complexity: str  # easy, medium, hard


FAILURE_CATALOG: List[FailureCase] = [
    FailureCase("Retrieval", "Vocabulary mismatch",
        "Query uses different terms than source document",
        "user says 'vacation policy' but doc says 'PTO'",
        "high", "medium"),
    FailureCase("Retrieval", "Cross-document reasoning",
        "Answer requires combining info from 2+ documents not co-retrieved",
        "What SSO features are planned for the enterprise onboarding?",
        "high", "hard"),
    FailureCase("Retrieval", "Temporal confusion",
        "Retriever returns outdated version of a document",
        "security policy (returns v1 instead of v2)",
        "high", "medium"),
    FailureCase("Generation", "Hallucination",
        "LLM generates facts not present in context",
        "adds specific dates or numbers not in retrieved chunks",
        "high", "medium"),
    FailureCase("Generation", "Under-specification",
        "Answer is too vague, misses key details from context",
        "says 'follow the policy' instead of quoting specifics",
        "medium", "easy"),
    FailureCase("Generation", "Conflation",
        "Merges information from unrelated chunks incorrectly",
        "mixes PTO policy details with expense policy",
        "high", "medium"),
    FailureCase("Citation", "Missing citations",
        "Correct answer but no source references",
        "states facts without [n] markers",
        "medium", "easy"),
    FailureCase("Citation", "Wrong citation",
        "Citation points to chunk that doesn't support the claim",
        "[2] cited for a fact only present in [1]",
        "medium", "medium"),
    FailureCase("Coverage", "Knowledge gap",
        "Question about topic not in the knowledge base",
        "cryptocurrency treasury policy",
        "low", "hard"),
    FailureCase("Coverage", "Permission gap",
        "Relevant doc exists but user lacks access",
        "salary bands query from engineering employee",
        "medium", "easy"),
]

# Summary table
print("Failure Analysis Catalog")
print("=" * 90)
df_failures = pd.DataFrame([
    {
        "Category": f.category,
        "Subcategory": f.subcategory,
        "Impact": f.impact,
        "Fix": f.fix_complexity,
        "Example": f.example_query[:45],
    }
    for f in FAILURE_CATALOG
])
print(df_failures.to_string(index=False))

# Improvement roadmap
print("\n\nImprovement Roadmap (prioritized by impact × fix ease)")
print("=" * 70)
ROADMAP = [
    ("P0 — Quick wins",     ["Add query expansion for vocabulary mismatch",
                              "Enforce citation [n] in system prompt",
                              "Add freshness weighting to retrieval scoring"]),
    ("P1 — Medium effort",  ["Implement reranking with cross-encoder",
                              "Add hallucination detection post-generation",
                              "Build feedback loop: thumbs up/down on answers"]),
    ("P2 — Strategic",      ["Multi-hop retrieval with iterative search",
                              "Fine-tune embedding model on enterprise corpus",
                              "Real-time connector sync for chat/email sources"]),
]
for priority, items in ROADMAP:
    print(f"\n  {priority}:")
    for item in items:
        print(f"    • {item}")

print("\n\n→ Start with P0 items — they address the highest-impact failures")
print("  with the lowest engineering effort.")

## 🏋️ Exercise 1 — Build a per-category evaluation dashboard

Group the evaluation queries by category (factual, multi-source, comparison,
multi-hop, abstention) and compute retrieval metrics per group. Which category
has the lowest MRR@10? Propose a specific improvement for that category.

In [ ]:
# YOUR CODE HERE
# Hint: use EVAL_SET categories and the metric functions defined above
# per_category_metrics = defaultdict(list)
# for eq in EVAL_SET:
#     ranking = simulate_hybrid_ranking(eq)
#     ...

## 🏋️ Exercise 2 — Design an automated regression test

Create a `run_regression_test` function that:
1. Runs retrieval on all eval queries
2. Asserts MRR@10 >= 0.7 and Recall@5 >= 0.8
3. Returns a pass/fail report with details on any regressions

This should be runnable as a CI check after any pipeline changes.

In [ ]:
# YOUR CODE HERE
# def run_regression_test(
#     eval_set: List[EvalQuery],
#     retriever_fn,
#     mrr_threshold: float = 0.7,
#     recall_threshold: float = 0.8,
# ) -> Dict:
#     ...

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Three evaluation dimensions**: retrieval (MRR, NDCG, Recall), generation (faithfulness, relevance), citation (precision, recall) |
| 2 | **Hybrid retrieval** (dense + BM25 + RRF) consistently outperforms single-method approaches |
| 3 | **LLM-as-judge** provides scalable answer quality evaluation without expensive human annotation |
| 4 | **Citation accuracy** is a distinct metric — high faithfulness doesn't guarantee proper source attribution |
| 5 | **Latency is dominated by LLM generation** (~90%) — retrieval is fast, generation is the bottleneck |
| 6 | **Failure categorization** enables prioritized improvement: fix high-impact, low-effort issues first |

## What's Next

You've built and evaluated a complete enterprise search engine! Next steps:
- **Scale up**: Add more documents, implement a real vector database
- **Production**: Add observability, caching, and feedback loops
- **Explore**: Try other labs in the Castalia series for more applied AI projects

## References

1. Thakur, N. et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models*. NeurIPS 2021.
2. Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. NeurIPS 2023.
3. Es, S. et al. (2023). *RAGAS: Automated Evaluation of Retrieval Augmented Generation*. arXiv:2309.15217.
4. OpenAI (2024). *Pricing*. https://openai.com/pricing